# 🔬 NCKH 2026: Generative AI in Synthetic Malware Generation
### Pipeline thử nghiệm 3 mô hình sinh (GAN, WGAN-GP, Diffusion) và Đánh giá 6 tiêu chí theo bài báo IEEE Access 2025

Notebook này tích hợp toàn bộ quy trình từ:
1. **Tạo/Nạp dữ liệu vector đặc trưng Opcode (104D)**
2. **Huấn luyện 3 mô hình GenAI**:
   - **Vanilla Dense GAN** (Baseline)
   - **Conv1D WGAN-GP** (Wasserstein GAN với Gradient Penalty)
   - **1D U-Net Diffusion Model** (Modified DDPM với $T/2$ Truncation)
3. **Thực nghiệm & Đánh giá theo 6 tiêu chí khoa học**:
   - Metric 1: Binary Classification (Real vs Synthetic)
   - Metric 2: t-SNE Visualization 2D
   - Metric 3: Synthetic-Only Training Evaluation
   - Metric 4: Fidelity Score
   - Metric 5: Threshold-Based Augmentation (Threshold $t = 0.95$)
   - Metric 6: Robustness & Cosine Similarity

In [ ]:
# Cài đặt các thư viện phụ thuộc (chạy nếu ở trên Google Colab)
!pip install torch torchvision scikit-learn matplotlib seaborn gensim tqdm

In [ ]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt
import torch

# Kiểm tra thiết bị tính toán (GPU / CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[*] Thiết bị thực thi: {device}")

## 1. Khởi tạo Dữ liệu Giả lập Benchmark (104 Chiều, 5 Họ Malware, Mất cân bằng lớp)

In [ ]:
from src.data_generator import generate_synthetic_benchmark_dataset, MALWARE_FAMILIES
from sklearn.model_selection import train_test_split

X_all, y_all, family_map = generate_synthetic_benchmark_dataset(n_features=104, random_seed=42)
print(f"Tổng số mẫu: {len(X_all)} | Số chiều vector embedding: {X_all.shape[1]}")
for idx, name in family_map.items():
    print(f" - Họ {idx} ({name}): {np.sum(y_all == idx)} mẫu")

# Chia tập Train và Test (bảo toàn phân bố lớp)
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.25, random_state=42, stratify=y_all
)
print(f"Kích thước tập Train: {len(X_train)} | Tập Test: {len(X_test)}")

## 2. Huấn luyện 3 Mô hình Sinh (Focus trên họ phức tạp Expiro.BK)

In [ ]:
from src.training.train_models import train_gan_model, train_wgan_gp_model, train_diffusion_model

target_family_idx = 3 # Expiro.BK
expiro_real_data = X_train[y_train == target_family_idx]
num_syn_samples = 200

# 1. Vanilla Dense GAN
print("[*] Huấn luyện Model 1: Vanilla GAN...")
gan_G, _ = train_gan_model(expiro_real_data, epochs=30, batch_size=16, latent_dim=64, device=device)
gan_G.eval()
with torch.no_grad():
    z = torch.randn(num_syn_samples, 64, device=device)
    syn_gan = gan_G(z).cpu().numpy()

# 2. Conv1D WGAN-GP
print("[*] Huấn luyện Model 2: Conv1D WGAN-GP...")
wgan_G, _ = train_wgan_gp_model(expiro_real_data, epochs=30, batch_size=16, latent_dim=64, device=device)
wgan_G.eval()
with torch.no_grad():
    z = torch.randn(num_syn_samples, 64, device=device)
    syn_wgan = wgan_G(z).cpu().numpy()

# 3. 1D U-Net Diffusion Model (Modified T/2)
print("[*] Huấn luyện Model 3: 1D U-Net Diffusion...")
diff_net, diffusion = train_diffusion_model(expiro_real_data, epochs=30, batch_size=16, timesteps=1000, device=device)
diff_anchor = torch.tensor(expiro_real_data, dtype=torch.float32, device=device)
syn_diff = diffusion.sample(diff_net, diff_anchor, num_samples=num_syn_samples).cpu().numpy()

print("[+] Hoàn thành sinh mẫu cho cả 3 mô hình!")

## 3. Thực nghiệm & Đánh giá Kết quả (6 Metrics)

In [ ]:
from src.evaluation.metrics import (
    evaluate_binary_classification,
    evaluate_threshold_augmentation,
    compute_cosine_robustness
)
from src.evaluation.visualize import plot_tsne_comparison, plot_augmentation_delta

# Metric 1: Binary Classification
bin_gan = evaluate_binary_classification(expiro_real_data, syn_gan)
bin_wgan = evaluate_binary_classification(expiro_real_data, syn_wgan)
bin_diff = evaluate_binary_classification(expiro_real_data, syn_diff)

print("--- Metric 1: Binary Classification (F1 Thật vs Giả) ---")
print(f"GAN:       RF={bin_gan['RandomForest']:.3f}, SVM={bin_gan['SVM']:.3f}, MLP={bin_gan['MLP']:.3f}")
print(f"WGAN-GP:   RF={bin_wgan['RandomForest']:.3f}, SVM={bin_wgan['SVM']:.3f}, MLP={bin_wgan['MLP']:.3f}")
print(f"Diffusion: RF={bin_diff['RandomForest']:.3f}, SVM={bin_diff['SVM']:.3f}, MLP={bin_diff['MLP']:.3f}")

# Metric 2: t-SNE Plot
plot_tsne_comparison(expiro_real_data, syn_gan, syn_wgan, syn_diff, family_name="Expiro.BK", save_path="results/tsne_comparison.png")

In [ ]:
# Hiển thị ảnh t-SNE trực tiếp trong notebook
from IPython.display import Image
Image("results/tsne_comparison.png")

In [ ]:
# Metric 5: Threshold Augmentation (Ngưỡng t = 0.95)
syn_dict = {3: syn_diff}
aug_res = evaluate_threshold_augmentation(X_train, y_train, X_test, y_test, syn_dict, threshold=0.95)
print("--- Metric 5: Tăng cường dữ liệu có điều kiện (Random Forest) ---")
print(f"F1 Baseline (Trước Augment):  {aug_res['baseline_overall_f1']:.4f}")
print(f"F1 Augmented (Sau Augment):   {aug_res['augmented_overall_f1']:.4f}")
print(f"Cải thiện (Delta F1):         +{aug_res['f1_improvement']:.4f}")

# Metric 6: Cosine Robustness Deviation
sim_gan = compute_cosine_robustness(expiro_real_data, syn_gan)
sim_wgan = compute_cosine_robustness(expiro_real_data, syn_wgan)
sim_diff = compute_cosine_robustness(expiro_real_data, syn_diff)
print("\n--- Metric 6: Cosine Similarity Median Deviation (Càng nhỏ càng giống Real) ---")
print(f"GAN Deviation:       {sim_gan['median_deviation']:.4f}")
print(f"WGAN-GP Deviation:   {sim_wgan['median_deviation']:.4f}")
print(f"Diffusion Deviation: {sim_diff['median_deviation']:.4f}")